# Selection robustness audit

This notebook audits selected-index stability across frozen selection seeds. It does not change GraphCov source code, train models, or use official test data for selection.

In [1]:
from pathlib import Path
import hashlib, json, itertools
import numpy as np

ROOT = Path('/project/prj-sis01/xuxiaoyu/reliability_medmnistc_ab')
SELECTION_ROOT = ROOT / 'artifacts' / 'selections'
OUT = ROOT / 'results' / 'summary_tables'
OUT.mkdir(parents=True, exist_ok=True)
DATASETS = ['organsmnist', 'organamnist', 'pathmnist', 'tissuemnist', 'bloodmnist']
RATIOS = ['ratio_002', 'ratio_005']
METHODS = ['random', 'el2n_top', 'forgetting', 'eva', 'facility', 'fps', 'herding', 'graph_a2']
PILOT_SEEDS = [1, 42, 2026]
SOURCE_ROOTS = {
    ('graph_a2', 1): Path('/project/prj-sis01/xuxiaoyu/graph_select_table1_original/graph_a2_selection_seeds_2026_1_train42/results/selection_seed_1'),
    ('graph_a2', 42): Path('/project/prj-sis01/xuxiaoyu/graph_select_table1_original/results/table1_graph_a2_seed42'),
    ('graph_a2', 2026): Path('/project/prj-sis01/xuxiaoyu/graph_select_table1_original/graph_a2_selection_seeds_2026_1_train42/results/selection_seed_2026'),
    ('random', 1): Path('/project/prj-sis01/xuxiaoyu/graph_select_table1_original/random_selection_seed1_train42/results'),
    ('random', 42): Path('/project/prj-sis01/xuxiaoyu/graph_select_table1_original/results/table1_random_seed42'),
    ('random', 2026): Path('/project/prj-sis01/xuxiaoyu/graph_select_table1_original/random_selection_seed2026_train42/results'),
}

In [2]:
def sha256_indices(a):
    return hashlib.sha256(np.asarray(a, dtype=np.int64).tobytes()).hexdigest()

def find_index_file(dataset, ratio, method, seed):
    source = SOURCE_ROOTS.get((method, seed))
    if source is not None and source.exists():
        ratio_token = ratio.replace('ratio_', 'ratio').replace('002', '0.02').replace('005', '0.05')
        matches = [p for p in source.rglob('selected_indices.npy') if dataset in p.parts and ratio_token in p.parent.name]
        if matches:
            return sorted(matches)[0]
    candidates = [
        SELECTION_ROOT / dataset / ratio / method / f'seed_{seed}' / 'selected_indices.npy',
        SELECTION_ROOT / dataset / ratio / method / f'selection_seed_{seed}' / 'selected_indices.npy',
    ]
    return next((p for p in candidates if p.exists()), None)

def jaccard(a, b):
    a, b = set(np.asarray(a).tolist()), set(np.asarray(b).tolist())
    return len(a & b) / len(a | b) if a or b else 1.0

rows = []
missing = []
for ds, ratio, method in itertools.product(DATASETS, RATIOS, METHODS):
    loaded = {}
    for seed in PILOT_SEEDS:
        path = find_index_file(ds, ratio, method, seed)
        if path is None:
            missing.append({'dataset': ds, 'ratio': ratio, 'method': method, 'seed': seed})
            continue
        loaded[seed] = np.load(path, allow_pickle=False)
    for s1, s2 in itertools.combinations(sorted(loaded), 2):
        rows.append({'dataset': ds, 'ratio': ratio, 'method': method, 'seed_a': s1, 'seed_b': s2, 'jaccard': jaccard(loaded[s1], loaded[s2]), 'size_a': int(len(loaded[s1])), 'size_b': int(len(loaded[s2])), 'sha_a': sha256_indices(loaded[s1]), 'sha_b': sha256_indices(loaded[s2])})

(OUT / 'selection_robustness_pairwise.json').write_text(json.dumps(rows, indent=2))
(OUT / 'selection_robustness_missing.json').write_text(json.dumps(missing, indent=2))
print({'pairwise_rows': len(rows), 'missing_cells': len(missing), 'output': str(OUT)})

{'pairwise_rows': 60, 'missing_cells': 180, 'output': '/project/prj-sis01/xuxiaoyu/reliability_medmnistc_ab/results/summary_tables'}


## Interpretation rule

Low Jaccard means the selector is sensitive to selection randomness; it is not evidence of downstream failure by itself. Downstream clean and MedMNIST-C evaluation must reuse each saved index set and one final checkpoint per training seed. No corruption-specific retraining is allowed.